# This noteboook aims to do an initial analysis of ERA5 and ERA5land Tmax data for identifying heatwave events over europe 

In [5]:
# ------------------------------------- Used libraries --------------------------------------------
import numpy as np
from matplotlib import pyplot as plt
from netCDF4 import Dataset as ncread
import xarray as xr 
from scipy.stats import linregress
from datetime import datetime, timedelta
import pandas as pd
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import os
import re
import ast
import xesmf as xe 
from matplotlib.backends.backend_pdf import PdfPages
import sys
# Add the path to the 'src' directory (or the root directory of your package)
sys.path.append('..') 
from functions_inputs import data_preprocess 
from functions_inputs.features_labels import *



# General purpose notebook

This notebook is used to create the inputs and labels of the DL model. The inputs are standardized anomalies of the local-scale variables (swvl[1,2,3], SPEI, SPI) and large-scale variables (g500, g200, psl). Any other variables can be included. 

The lable sof the DL model are a daily binary classification of temeprature extremes resulting from a percentile definition based on Peskins & Alexander: On the measurement of Heat Waves: https://journals.ametsoc.org/view/journals/clim/26/13/jcli-d-12-00383.1.xml.

Blocks:

* Anomalies: to compute the anomalies of the features for the DL model. 
* Detrend data: block to linearly detrend the features. This block is not used in the paper. 
* Detext Extreme Events: Key block to save files with standardized anomalies and binary classification of extreme events. 
* Preparation of time-lagged features in the input data Local-Scale: block to create netCDF files with lagged standardized inputs. These are the final features to be passeed to the DL model. 


* Large Scale Data: Contains three blocks with the same objectives as the previous one for the local-scale data; Detrend, Anomalies, Lagged-Data. Additionally, it has a forth block that is used to re-grid the girdded fields to reduce the number of parameters that the DL model needs resulting from the large-scale fields. 


LOESS fit is passed to the percentile values based on Mahlstein et al. 2015

# Anomalies

Code to save the anomalies for each sat in a netCDF file. Anomalies are standarized using:

$$standarized_{anomaly} = \frac{anomaly-\mu}{\sigma}$$

These anomalies will serve as input for the DL model

In [5]:

sites = ['cordoba','hannover','stockholm','lyon','belgrado','marrakech']
outputpath = "/gpfs/scratch/bsc32/bsc167965/tfm_data/era5_land/anomalies/"


variables = ['tasmax','tasmin','swvl1','swvl2','swvl3']

for site in sites: 

    path = f'/gpfs/scratch/bsc32/bsc167965/tfm_data/era5_land/variables_era5land_data_{site}_1950_2024.nc'
    ds = xr.open_dataset(path) #open dataset

    for variable in variables: 
    
        #extract anomalies with climatology computed using loess fitting. Select reference period for the climatology computation 
        standarized_anomalies = Compute_anomalies(ds,variable,'1950','2000') #call Compute_anomalies function

        #Save the two type anomalies for the variable
        ds[f'{variable}_anomalies'] = standarized_anomalies
        ds[f'{variable}_anomalies'].attrs = {
            'long_name':'Anomalies with loess climatology',
            'description':'These anomalies have been computed with a loess fit on a raw computation of the climatology'
        }
     
    #Save new netCDF file including the anomalies computed with the two methods 
    ds.to_netcdf(outputpath+f"std_changed_variables_era5land_data_with_anomalies_{site}_1950_2024.nc")

    
    

# Detrend Data

In [ ]:
# Sites to process
sites = ['cordoba', 'hannover', 'stockholm', 'lyon', 'belgrado', 'marrakech']

# Loop through each site

for site in sites:

    path = f'/gpfs/scratch/bsc32/bsc167965/tfm_data/era5_land/variables_era5land_data_{site}_1950_2024.nc'
    ds = xr.open_dataset(path).sel(time=slice('1950', '2024'))
    
    # Create a copy for detrended data
    ds_detrended = xr.Dataset()
    
    # Loop through each variable and detrend
    for var in ds.data_vars:
        da = ds[var]

        time_index = xr.DataArray(
            np.arange(len(da.time)),
            dims=['time_index'],
            coords={'time_index': np.arange(len(da.time))},
            name='time_index'
        )

        
        da_with_index = xr.DataArray(
            da.values,
            dims=['time_index', *da.dims[1:]],
            coords={
                'time_index': time_index,
                **{dim: da[dim] for dim in da.dims[1:]}
            }
        )
        
        # 4. Now perform the fit using time_index dimension
        fit = da_with_index.polyfit(dim='time_index', deg=1, skipna=True)
        
        # 5. Evaluate trend
        trend_line = xr.polyval(da_with_index['time_index'], fit['polyfit_coefficients'])
        
        # 6. Detrend and add to original dataset (swap back to time dimension)
        ds_detrended[f'{var}_detrended'] = (da.dims, (da.values - trend_line.values))
        
        # --- Verification ---
        slope = fit['polyfit_coefficients'].sel(degree=1)
        print(f"Detrending slope: {slope.item()}")
    
    # Save detrended dataset to NetCDF
    output_path = f'/gpfs/scratch/bsc32/bsc167965/tfm_data/era5_land/detrended_variables_era5land_data_{site}_1950_2024.nc'
    ds_detrended.to_netcdf(output_path)

# Detect Extreme Events

### Heatwave Detection and Anomaly Computation

This block performs the detection of heatwave events and the computation of soil water anomalies for multiple locations.  
The workflow includes the following steps:

1. **Data Loading**  
   - Load soil water variables (`swvl1`, `swvl2`, `swvl3`) from ERA5-Land.  
   - Load maximum temperature (`tasmax`) from ERA5 observational data.  
   - Time period: 1950–2023 (reference period: 1950–2000).  

2. **Reference Percentile and Climatology**  
   - Compute the climatology and the 90th percentile for `tasmax` over the reference period.  
   - Apply a LOESS smoothing to remove noise in the percentile estimation.  

3. **Preprocessing**  
   - Remove leap day (February 29) to ensure consistency.  
   - Convert smoothed percentiles into an `xarray.DataArray` for compatibility.  

4. **Heatwave Detection**  
   - Use the smoothed percentile and a 5-day moving window climatology.  
   - Events are defined as days above the 90th percentile.  
   - Save a binary mask (1 = extreme, 0 = non-extreme) into a NetCDF file.  

5. **Anomaly Computation**  
   - For each soil water variable, compute standardized anomalies relative to the reference climatology.  
   - Save anomalies alongside the heatwave classification into a NetCDF file.  

6. **Extreme Event Metrics**  
   - Compute metrics such as:  
     - Duration, frequency, and cumulative intensity of heatwaves.  
     - Counts and percentages of normal vs. extreme days per period.  
   - Write metrics to a `.txt` file for each site.  

**Outputs generated per site:**  
- NetCDF file with anomalies and heatwave classification.  
- Text file with statistics of detected heatwave events.  


In [5]:
import functions_HW
import loess_functions

sites = ['cordoba', 'hannover', 'stockholm', 'lyon', 'belgrado', 'marrakech']
#variables = ['swvl1','swvl2','swvl3']
variables = ['spei_hg_3']

variable_extreme = 'tasmax'

#This loop wil create a csv file of dates and duration of detected extreme events for the full period contained in the netCDF file. 
#Additionally, it creates a .txt file with metrics for the extreme events in the diferent locations defined in sites. 

for site in sites:

    path =  f'/gpfs/projects/bsc32/bsc167965/observational_TX/spei_tasmax_{site}_hg_3.nc'
    #path = f'/gpfs/scratch/bsc32/bsc167965/tfm_data/era5_land/variables_era5land_data_{site}_1950_2024.nc'
    ds = xr.open_dataset(path).sel(time=slice('1950','2023'))

    # climatology, percentile, and window
    #the climatology is computed for the reference period only, and same for the percentile 
    climatology,percentile,clim_window,full_window,loess_clim = functions_HW.Compute_window_percentile_reference_period(ds,'tasmax',0.9,5,'1950','2000')

    #remove 29th of febraury and original variables--------------------------------------------------------------------------
    if 366 in percentile.dayofyear:
        ds = ds.sel(time=~((ds.time.dt.month == 2) & (ds.time.dt.day == 29)))
    
    #apply loess fit to the raw_percentile 
    loess_percentile = loess_functions.loess_ts(percentile, na_rm=True,  window=30,  degree=1)
    
    #convert the loess_percentile data into xarray to use it as input in detect_HW() funtion
    
    # Create a new xarray.DataArray with the same format as percentile
    loess_percentile_xarray = xr.DataArray(
        loess_percentile, 
        dims=["dayofyear"], 
        coords={"dayofyear": percentile.dayofyear},  # Use the same dayofyear coordinate
        name="loess_percentile"
    )

    #Call detect heatwave function for the max temperature data, either ERA5 or ERA5 land 
    # DateTime_time_out_detect: datetime used in detect_HW_functions. Will be used in Compute_metrics function to get the frequencies of individual periods of time 
    HW_mask,HW_intensity,DateTime_time_out_detect = functions_HW.detect_HW(ds,variable_extreme,loess_percentile_xarray,3,site) 
    #HW_mask will give a number of True values around 10% if the 90th percentile is used 

   # Call the anomalies function to create netcdf files that have the standarized anomalies and 0s and 1s for th extreme event classification -------------------------------

    output_path = '/path/to/save/anomalies/and/extreme/event/labels/'
    file_path = f'spei3_{site}_standarized_anomalies_and_extreme_detection.nc'

    ds_save = ds.copy()
    
    HW_mask_0_1 = HW_mask.astype(int) #convert boolean mask to 0s and 1s 


    #Compute the anomalies for all the ERA5 land data and save in the ds file ------------------------------------------------------
    for var in variables:
        anomalies = functions_HW.Compute_anomalies(ds,var,'1950','2000')   

        #Save the two type anomalies for the variable
        ds_save[f'{var}_anomalies'] = xr.DataArray(
            anomalies.values,
            coords={'time': ds.time},
            dims=['time']
        )
        ds_save[f'{var}_anomalies'].attrs = {
            'long_name':f'{var} standarized anomalies with loess fit climatology',
            'description':f'These anomalies for ERAland {var} have been computed with a raw computation of the climatology to which a loess fitting is applied afterwards'
        }

    # -------------------------------------------------------------------------------------------------

    #Save the extreme classification
    ds_save[f'{variable_extreme}_extreme_classification'] = (('time'),HW_mask_0_1)
    ds_save[f'{variable_extreme}_extreme_classification'].attrs = {
        'extreme detection with percentile 90 and detrended data':'Anomalies with 5-day window fit climatology',
        'description': ("1 stands for extreme event in that time and space point, 0 stands for non-extreme. Detection was done usnig a smoothed loess fit percentile and a moving 5-day window climatology. Both percentile and climatology are computed only for the reference period 1971-2000." )
    }

    ds_save.to_netcdf(output_path+file_path)

    # ------------------------------------------------------------------------------------------------------------------

    #Move to metric computation using Compute_metrics funtion 
    
    #Call function to compute metrics of the detected extreme events 
    duration,frequency,frequencies_selected_periods,max_intensity,mean_intensity,cumulative_intensity,period_percentages,period_counts = functions_HW.Compute_metrics(HW_mask,HW_intensity,DateTime_time_out_detect)

    #Write in a pdf file the metrics of the extreme events detected 
    
    with open(f'HW_dates_detected_and_metrics/spei3_heatwave_stats_{site}.txt', 'w') as file:
        # Write each statistic to the file
        file.write(f'Site:{site} \n')
        file.write(f'Time period: 1950-2024 \n')
        file.write(f'Reference time period for 90th percentile computation: 1971-2000 \n')
        file.write(f'Minimum duration for heat wave event: 3 days \n')    
        file.write(f'Max duration of event: {duration.max()}\n')
        file.write(f'Min duration of event: {duration.min()}\n')
        file.write(f'Frequency in the grid-point: {frequency}\n')
        file.write(f'Frequency in the grid-point for the selected periods: {frequencies_selected_periods}\n')
        file.write(f'Count normal and extreme days per period: {period_counts}\n')
        file.write(f'Percentages normal and extreme days per period: {period_percentages}\n')
        file.write(f"Max value of max intensity: {max_intensity.max()}\n")
        file.write(f'Mean intensity: {mean_intensity.max()}\n')
        file.write(f'Max accumulated intensity: {cumulative_intensity.max()}\n')
        file.write(f'Min accumulated intensity: {cumulative_intensity.min()}\n')
        
    

Heatwave events saved to 'heatwave_events_1950_2024_cordoba.csv'
All days 1950-2000 = 18615
{'1950-2000': {'HW_days': np.int64(1897), 'All_days': np.int64(18615)}, '1971-2000': {'HW_days': np.int64(1293), 'All_days': np.int64(10950)}, '2001-2024': {'HW_days': np.int64(1589), 'All_days': np.int64(8395)}}
{'1950-2000': {'HW_days_percent': np.float64(10.190706419554123)}, '1971-2000': {'HW_days_percent': np.float64(11.808219178082192)}, '2001-2024': {'HW_days_percent': np.float64(18.92793329362716)}}
{'1950-2000': 732, '1971-2000': 486, '2001-2024': 534}
Heatwave events saved to 'heatwave_events_1950_2024_hannover.csv'
All days 1950-2000 = 18615
{'1950-2000': {'HW_days': np.int64(1899), 'All_days': np.int64(18615)}, '1971-2000': {'HW_days': np.int64(1218), 'All_days': np.int64(10950)}, '2001-2024': {'HW_days': np.int64(1449), 'All_days': np.int64(8395)}}
{'1950-2000': {'HW_days_percent': np.float64(10.201450443190975)}, '1971-2000': {'HW_days_percent': np.float64(11.123287671232877)}, '20

IndexError: index 22 is out of bounds for axis 0 with size 22

 # Preparation of time-lagged features in the input data Local-Scale

Now we will create netCDF files with anomalies data for the different ERA5 and ERA5land variables so that the input is prepared for the Neural Network

### Creation of Lagged Features for Neural Network Input

This block generates lagged versions of climate anomalies and extreme event classifications, preparing the data for machine learning (e.g., artificial neural networks).  

**Workflow:**

1. **Setup**  
   - Define study sites: Córdoba, Hannover, Stockholm, Lyon, Belgrado.  
   - Define the variables of interest:  
     - Temperature anomalies (`tasmax_anomalies`, `tasmin_anomalies`).  
     - Soil water anomalies (`swvl1_anomalies`, `swvl2_anomalies`, `swvl3_anomalies`).  
     - Extreme classification based on maximum temperature (`tasmax_extreme_classification`).  
   - Specify a dictionary (`lag_dict`) that maps each variable to the number of time lags to compute (e.g., soil water up to 7 days, temperature up to 3 days).  

2. **Lagged Feature Creation**  
   - For each site:  
     - Load the NetCDF file containing standardized anomalies and extreme classification.  
     - Call the `create_lagged_features_multiple()` function, which:  
       - Iterates through each variable.  
       - Creates new variables shifted by the specified lag days.  
       - Stores them in the dataset with descriptive metadata.  
     - Save the enriched dataset (original variables + lagged features) as a new NetCDF file.  

3. **Purpose**  
   - The resulting datasets provide **lagged predictors** that will be used as input features for neural network models.  
   - This enables the model to capture **temporal dependencies** (e.g., soil moisture or temperature conditions in the previous 3–7 days) that may influence heatwave occurrences.  

**Outputs per site:**  
- NetCDF file containing the original anomalies, extreme classifications, and newly generated lagged features.  


In [3]:


#Define sites ERA5land
sites = ['cordoba','hannover','stockholm','lyon','belgrado']
# Define variables and their lags
variables = [
 'tasmax_anomalies',
 'tasmin_anomalies',
 'swvl1_anomalies',
 'swvl2_anomalies',
 'swvl3_anomalies',
 'tasmax_extreme_classification']

#dictionary indicating number of days of lagged-data for each of the variables 
lag_dict = { 
 'tasmax_anomalies': [1, 2, 3],
 'tasmin_anomalies': [1, 2, 3],
 'swvl1_anomalies': [1, 2, 3, 4, 5, 6, 7],
 'swvl2_anomalies': [1, 2, 3, 4, 5, 6, 7],
 'swvl3_anomalies': [1, 2, 3, 4, 5, 6, 7],
 'tasmax_extreme_classification': [1,2,3] }

#Loop over the different locations

for site in sites:
    #path where netCDF file is stored. Contains ERA5 land anomalies and extreme event classification based on tasmax
    input_path = f'/your/path/to/anomalies/and/event/detection/data/90p_{site}_standarized_anomalies_and_extreme_detection.nc'
    ds = xr.open_dataset(input_path)
    #path where new netCDF file is saved
    out_path = f'/oyur/path/to/save/the/lagged/data/90p_{site}_lagged_standarized_anomalies_and_extreme_detection.nc'
    # Apply the function
    ds_lagged = data_preprocess.create_lagged_features_multiple(ds,variables,lag_dict,'lagged_era5_land_')
    ds_lagged.to_netcdf(out_path)


# Large-Scale data ----------------------------------------------------------------------------------------------

## Detrend

In [ ]:
import xarray as xr
import data_preprocess

variables = ['g500','g200', 'psl']
variables = ['g500']

regrid = True

for variable in variables:
    input_path = f'/gpfs/scratch/bsc32/bsc167965/tfm_data/era5/{variable}_era5_data_full_domain_1950_2024.nc'
    output_path = f'/gpfs/scratch/bsc32/bsc167965/tfm_data/era5/regridded_detrended/{variable}_regridded_detrended_1x1_era5_data_full_domain_1950_2024.nc'

    # Open the dataset
    ds = xr.open_dataset(input_path)
    
    # Get the variable
    da = ds[variable]

    # Fit linear trend per grid point (over time)
    trend_coeffs = da.polyfit(dim='time', deg=1)
    trend_fit = xr.polyval(da['time'], trend_coeffs.polyfit_coefficients)

    # Subtract the trend (detrend)
    da_detrended = da - trend_fit

    # Replace original variable with detrended version
    ds[variable] = da_detrended

    # Optionally regrid
    if regrid:
        print(f"Regridding {variable}...")
        ds_regridded = data_preprocess.regridded_dataset(ds, variable)

    # Save to NetCDF
    #ds_regridded.to_netcdf(output_path)

    print(f"Regridding and detrending {variable} complete")

## Regridding ERA5 data

To make the ERA5 data files more easy to handle, we create a regridder from the original resolution of 0.25ºx0.25º to 1ºx1º

In [2]:
import data_preprocess

variables = ['g500','g200','psl']

for variable in variables:

    input_path = f'/gpfs/scratch/bsc32/bsc167965/tfm_data/era5/{variable}_era5_data_full_domain_1950_2024.nc'
    output_path = f'/gpfs/scratch/bsc32/bsc167965/tfm_data/era5/regridded/{variable}_regridded_1x1_era5_data_full_domain_1950_2024.nc'

    # Open the dataset
    ds = xr.open_dataset(input_path)  # Replace with your actual file
    
    ds_regridded = data_preprocess.regridded_dataset(ds,variable)
    # Save the new dataset
    ds_regridded.to_netcdf(output_path)
    
    print(f"Regridding {variable} complete")

Regridding g200 complete
Regridding psl complete


# Anomalies regridded data ERA5

Next block creates files that contain the original data for the different variables and the standarized anomalies 

In [3]:
import functions_HW


outputpath = "/gpfs/scratch/bsc32/bsc167965/tfm_data/era5/anomalies/"

variables = ['g500']

variables = ['g200','psl']

for var in variables: 

    base_path_regridded = '/your/path/to/regridded/variables/'
    file_regridded = f'{var}_regridded_1x1_era5_data_full_domain_1950_2024.nc'

    #Open dataset
    ds = ds = xr.open_dataset(base_path_regridded+file_regridded) #.chunk({"time":500})

    # Remove 29th of february 
    ds = ds.sel(time=~((ds.time.dt.month == 2) & (ds.time.dt.day == 29)))


    #extract climatology, window-climatology, percentile
    #Can be computed for a certain reference period or for the full available time period
    anomalies = functions_HW.Compute_anomalies(ds,var,'1950','2000') #call Compute_anomalies function

    print(f'anomalies {var} computed, moving to creating netCDF to be saved')

    # Drop original variable 
    ds = ds.drop_vars(var)

    #Save the two type anomalies for the variable
    ds[f'{var}_anomalies'] = anomalies
    ds[f'{var}_anomalies'].attrs = {
        'long_name':'Anomalies with LOESS-fit climatology',
        'description':'These anomalies have been computed with a 30-day LOESS-fit of the climatology, simply taking the daily average'
    }

    #Save file only containing anomalies 
    ds.to_netcdf(outputpath+f"{var}_1x1_era5_data_only_anomalies_1950_2024.nc")
    print(f'{var} done and anomalies saved to netcdf')


    
    

anomalies g200 computed, moving to creating netCDF to be saved
g200 done and anomalies saved to netcdf
anomalies psl computed, moving to creating netCDF to be saved
psl done and anomalies saved to netcdf


# Lagged data regridded data ERA5

In [7]:
import data_preprocess


# Define variables and their lags
variables = [
 #'tasmax',
 #'tasmin',
 'g500',
 'g200',
 'psl']

#dictionary indicating number of days of lagged-data for each of the variables 
lag_dict = {
 #'tasmax': [1, 2, 3],
 #'tasmin': [1, 2, 3],
 'g200': [1,2,3],
 'g500': [1,2,3],
 'psl' : [1,2,3]
 }



#Loop over the different locations

for var in variables:
    #path where netCDF file is stored. Contains ERA5 land anomalies and extreme event classification based on tasmax
    input_path = f'/gpfs/scratch/bsc32/bsc167965/tfm_data/era5/anomalies/std_changed_{var}_1x1_era5_data_only_anomalies_1950_2024.nc'
    ds = xr.open_dataset(input_path)
    ds = ds.where(ds.time.dt.strftime('%Y-%m-%d') != '2024-07-31', drop=True)
    #path where new netCDF file is saved
    out_path = f'/gpfs/scratch/bsc32/bsc167965/tfm_data/era5/lagged_anomalies/std_changed_{var}_1x1_lagged_standarized_anomalies.nc'
    
    # Apply the function
    ds_lagged = data_preprocess.create_lagged_features_single(ds,var,lag_dict[var],'lagged_era5')

    ds_lagged.to_netcdf(out_path)

    ds.close()
    ds_lagged.close()

    print(f'{var} completed')


g500 completed
g200 completed
psl completed
